# DB28 Exploratory Analysis — From Counts to Insights
**Dataset:** T-100 Domestic Segment (DB28)

**Fields:** `YEAR, MONTH, CARRIER, ORIGIN, DEST, PASSENGERS, FLIGHTS, DISTANCE, RPM, ASM`

**Objective:** Move past `.describe()` into interpretive analytics — patterns, ratios, and insights that hint at business questions.

## 0) Setup

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

print('pandas:', pd.__version__)
print('numpy :', np.__version__)

## 1) Load Data
Set `PATH` to a small curated CSV extracted from T-100 Segment (domestic slice).

In [ ]:
# TODO: Update to your curated sample file path
PATH = "data/bts_samples/t100_segment_sample.csv"

# Read CSV; adjust dtypes as needed for your sample
dtype_map = {
    'YEAR': 'int16', 'MONTH': 'int8',
    'FLIGHTS': 'int32', 'PASSENGERS': 'int32',
    'DISTANCE': 'float32', 'RPM': 'float32', 'ASM': 'float32',
    'CARRIER': 'string', 'ORIGIN': 'string', 'DEST': 'string'
}
df = pd.read_csv(PATH, dtype=dtype_map, low_memory=False)
df.head(3)

## Ingest Examples (DB28 / T-100)
Use these patterns to bring raw BTS files into curated teaching CSVs.

**Option A — Local CSV already downloaded:** place raw file under `data/bts_ingest/` and trim to a sample.

**Option B — From BTS PREZIP URL:** download a ZIP, extract CSV, then trim.


In [ ]:
from notebooks._helpers.prepare_bts_sample import load_trim_save

# Example A: trim a locally available large CSV into a small teaching sample
# (Adjust src_csv to your raw DB28 CSV; adjust filters to 2025 Q1 or June 2025 as needed.)
src_csv = "data/bts_ingest/T100_segment_2025_01.csv"  # <-- update
dst_csv = "data/bts_samples/t100_segment_sample.csv"

columns = ["YEAR","MONTH","CARRIER","ORIGIN","DEST","FLIGHTS","PASSENGERS","DISTANCE","ASM","RPM"]
filters = {"YEAR":[2025], "MONTH":[1,2,3], "CARRIER": None}  # Q1 example; set to [6] for June

_ = load_trim_save(src_csv=src_csv, dst_csv=dst_csv, columns=columns, filters=filters)
print("Wrote sample:", dst_csv)

In [ ]:
# Example B: fetch from PREZIP (requires internet), extract, then trim
from notebooks._helpers.prepare_bts_sample import fetch_zip_to, extract_zip, load_trim_save
import os

# 1) Provide a valid PREZIP URL for DB28 (T-100 Segment) monthly data (CSV inside ZIP).
#    Get the URL from https://transtats.bts.gov/PREZIP/ (exact filenames vary by month/dataset).
prezip_url = "<PASTE_T100_SEGMENT_PREZIP_URL_ZIP>"  # <-- replace with actual ZIP URL
zip_path = "data/bts_ingest/t100_segment_2025_01.zip"
csv_dir = "data/bts_ingest"

try:
    fetch_zip_to(prezip_url, zip_path)
    extracted = extract_zip(zip_path, csv_dir)
    print("Extracted:", extracted)
    # 2) Pick the CSV we want (assume the first extracted CSV is the DB28 file)
    src_csv = next((p for p in extracted if p.lower().endswith(".csv")), None)
    if not src_csv:
        raise FileNotFoundError("No CSV found in ZIP. Check the PREZIP URL.")
    # 3) Trim and save a teaching sample
    columns = ["YEAR","MONTH","CARRIER","ORIGIN","DEST","FLIGHTS","PASSENGERS","DISTANCE","ASM","RPM"]
    filters = {"YEAR":[2025], "MONTH":[6], "CARRIER": None}  # Example: June 2025
    dst_csv = "data/bts_samples/t100_segment_sample.csv"
    load_trim_save(src_csv=src_csv, dst_csv=dst_csv, columns=columns, filters=filters)
    print("Wrote sample:", dst_csv)
except Exception as e:
    print("NOTE:", e)
    print("This example requires a valid PREZIP URL and internet access.")

## Ingest Examples (ASQP — On-Time Performance)
Goal: curate a small reliability-focused slice with delay and cancellation fields for 2025Q1 and June 2025.

**Typical columns:** YEAR, MONTH, CARRIER (or OP_CARRIER), ORIGIN, DEST, DEP_DELAY, ARR_DELAY, CANCELLED, DIVERTED, AIR_TIME


In [ ]:
from notebooks._helpers.prepare_bts_sample import load_trim_save

# ASQP Example A — Local CSV → trimmed sample (Q1 2025)
src_csv = "data/bts_ingest/ASQP_2025_01.csv"   # <-- update to your raw ASQP CSV
dst_csv = "data/bts_samples/asqp_sample.csv"

# Column names vary by vintage; adjust if your files use OP_UNIQUE_CARRIER or similar.
columns = ["YEAR","MONTH","CARRIER","ORIGIN","DEST","DEP_DELAY","ARR_DELAY","CANCELLED","DIVERTED","AIR_TIME"]
filters = {"YEAR":[2025], "MONTH":[1,2,3], "CARRIER": None}

_ = load_trim_save(src_csv=src_csv, dst_csv=dst_csv, columns=columns, filters=filters)
print("Wrote ASQP sample:", dst_csv)

In [ ]:
# ASQP Example B — PREZIP fetch → extract → trim (June 2025)
from notebooks._helpers.prepare_bts_sample import fetch_zip_to, extract_zip, load_trim_save

prezip_url = "<PASTE_ASQP_PREZIP_URL_ZIP>"   # e.g., monthly ASQP zip for 2025-06
zip_path = "data/bts_ingest/asqp_2025_06.zip"
csv_dir = "data/bts_ingest"

try:
    fetch_zip_to(prezip_url, zip_path)
    extracted = extract_zip(zip_path, csv_dir)
    print("Extracted:", extracted)

    src_csv = next((p for p in extracted if p.lower().endswith(".csv")), None)
    if not src_csv:
        raise FileNotFoundError("No CSV found in ZIP. Check the PREZIP URL.")

    columns = ["YEAR","MONTH","CARRIER","ORIGIN","DEST","DEP_DELAY","ARR_DELAY","CANCELLED","DIVERTED","AIR_TIME"]
    filters = {"YEAR":[2025], "MONTH":[6], "CARRIER": None}
    dst_csv = "data/bts_samples/asqp_sample.csv"
    load_trim_save(src_csv=src_csv, dst_csv=dst_csv, columns=columns, filters=filters)
    print("Wrote ASQP sample:", dst_csv)
except Exception as e:
    print("NOTE:", e)
    print("This example requires a valid PREZIP URL and internet access.")

## Ingest Examples (DB1B — O&D Survey)
Goal: curate a small demand/fare slice for 2025Q1 (DB1B releases are quarterly). Start with fare and distance fields.

**Typical columns:** YEAR, QUARTER, ORIGIN, DEST, CARRIER (optional), PASSENGERS, FARE, DISTANCE


In [ ]:
from notebooks._helpers.prepare_bts_sample import load_trim_save

# DB1B Example A — Local CSV → trimmed sample (2025 Q1)
src_csv = "data/bts_ingest/DB1B_2025_Q1.csv"   # <-- update to your raw DB1B CSV
dst_csv = "data/bts_samples/db1b_sample.csv"

columns = ["YEAR","QUARTER","ORIGIN","DEST","PASSENGERS","FARE","DISTANCE"]
filters = {"YEAR":[2025], "QUARTER":[1]}

_ = load_trim_save(src_csv=src_csv, dst_csv=dst_csv, columns=columns, filters=filters)
print("Wrote DB1B sample:", dst_csv)

In [ ]:
# DB1B Example B — PREZIP fetch → extract → trim (2025 Q1)
from notebooks._helpers.prepare_bts_sample import fetch_zip_to, extract_zip, load_trim_save

prezip_url = "<PASTE_DB1B_PREZIP_URL_ZIP>"   # e.g., quarterly DB1B zip for 2025 Q1
zip_path = "data/bts_ingest/db1b_2025_q1.zip"
csv_dir = "data/bts_ingest"

try:
    fetch_zip_to(prezip_url, zip_path)
    extracted = extract_zip(zip_path, csv_dir)
    print("Extracted:", extracted)

    src_csv = next((p for p in extracted if p.lower().endswith(".csv")), None)
    if not src_csv:
        raise FileNotFoundError("No CSV found in ZIP. Check the PREZIP URL.")

    columns = ["YEAR","QUARTER","ORIGIN","DEST","PASSENGERS","FARE","DISTANCE"]
    filters = {"YEAR":[2025], "QUARTER":[1]}
    dst_csv = "data/bts_samples/db1b_sample.csv"
    load_trim_save(src_csv=src_csv, dst_csv=dst_csv, columns=columns, filters=filters)
    print("Wrote DB1B sample:", dst_csv)
except Exception as e:
    print("NOTE:", e)
    print("This example requires a valid PREZIP URL and internet access.")

## 2) Basic Hygiene (quick checks)

In [ ]:
display(df.shape)
display(df.isna().sum().sort_values(ascending=False).head(10))
display(df.duplicated().sum())

# Example: guard against non-positive distances and zero denominators
df.loc[df['DISTANCE'] <= 0, 'DISTANCE'] = np.nan
df['ASM'].replace(0, np.nan, inplace=True)
df['FLIGHTS'].replace(0, np.nan, inplace=True)

df = df.dropna(subset=['DISTANCE','FLIGHTS','ASM'])

## 3) Analysis Focus
| Level | Question | Technique |
|:--|:--|:--|
| **A** | Which carriers dominate total passenger traffic? | Group & aggregate |
| **B** | What is the relationship between distance and load factor? | Derived metrics + scatter correlation |
| **C** | Which routes show under- or over-utilization? | KPI benchmarking + conditional filters |

### A. Carrier Market Shares

In [ ]:
carrier_pax = (
    df.groupby('CARRIER', dropna=False)['PASSENGERS']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)
carrier_pax['SHARE'] = carrier_pax['PASSENGERS'] / carrier_pax['PASSENGERS'].sum()
display(carrier_pax.head(10))

# Optional: simple bar chart for top carriers
ax = carrier_pax.head(10).plot(kind='bar', x='CARRIER', y='PASSENGERS', legend=False)
ax.set_title('Top Carriers by Total Passengers')
ax.set_xlabel('Carrier')
ax.set_ylabel('Passengers')
plt.tight_layout()
plt.show()

### B. Distance vs. Load Factor

In [ ]:
df['LOAD_FACTOR'] = df['RPM'] / df['ASM']

subset = df[['DISTANCE','LOAD_FACTOR']].dropna()
ax = subset.plot.scatter(x='DISTANCE', y='LOAD_FACTOR', alpha=0.3)
ax.set_title('Distance vs Load Factor')
plt.tight_layout()
plt.show()

subset.corr()

### C. Route Utilization Benchmark

In [ ]:
route_summary = (
    df.groupby(['ORIGIN','DEST'])
      .agg(PAX=('PASSENGERS','sum'),
           FLIGHTS=('FLIGHTS','sum'),
           DIST=('DISTANCE','mean'),
           ASM=('ASM','sum'),
           RPM=('RPM','sum'))
      .reset_index()
)
route_summary['LOAD_FACTOR'] = route_summary['RPM'] / route_summary['ASM']
mean_lf = route_summary['LOAD_FACTOR'].mean()
route_summary['PERF_VS_AVG'] = route_summary['LOAD_FACTOR'] - mean_lf

# Bottom & top performers
under = route_summary.sort_values('PERF_VS_AVG').head(5)
over  = route_summary.sort_values('PERF_VS_AVG', ascending=False).head(5)

display(under)
display(over)

## 4) Discussion Points
- Does higher distance always yield better load factors? Why or why not?
- What structural or market factors could explain route underperformance?
- How would seasonal demand or aircraft assignment affect these metrics?

## 5) Teaching Summary
1. **Aggregation → Ranking:** Market structure insight.
2. **Derived Ratios → Relationships:** Efficiency vs. scale.
3. **Benchmarking → Performance Classification:** Actionable insights.

## 6) Optional Extensions
- Add trend analysis by month to spot seasonality.
- Join with ASQP data to relate reliability to load factor.
- Calculate route yield if revenue/FARE is available.